## Project Aim

The primary objective of this study is to explore employee characteristics and discover hidden patterns within an organization's workforce through unsupervised machine learning techniques. By analyzing various factors, including employee demographics, job characteristics, job satisfaction, and career history, this project aims to identify distinct groups of employees with similar characteristics and working conditions.

To achieve this objective, clustering algorithms such as K-Means, DBSCAN, and Hierarchical Clustering are implemented to segment employees into meaningful groups without relying on predefined target labels. The performance of these models is evaluated using appropriate clustering metrics, including the Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Index, to assess the quality and separation of the resulting clusters.

Furthermore, employee attrition is examined as an additional variable to investigate whether different employee groups exhibit different turnover patterns. Based on the clustering results, the study seeks to identify meaningful workforce segments and provide data-driven insights and strategic recommendations to help the Human Resources department better understand employee needs, improve retention strategies, and support more effective workforce management.

## Importing Libraries

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

In [23]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "archive"

df_general = pd.read_csv(DATA_DIR / "general_data.csv")
df_employee = pd.read_csv(DATA_DIR / "employee_survey_data.csv")
df_manager = pd.read_csv(DATA_DIR / "manager_survey_data.csv")
df_inTime = pd.read_csv(DATA_DIR / "in_time.csv")
df_outTime = pd.read_csv(DATA_DIR / "out_time.csv")


## Reading Dataset

In [ ]:
df_general.head()
df_employee.head()
df_manager.head()
df_inTime.head()
df_outTime.head()

## Data inspection

In [17]:
# General
df_general.info()

df_general.describe(include='all')

df_general.isnull().sum()

df_general.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   str    
 2   BusinessTravel           4410 non-null   str    
 3   Department               4410 non-null   str    
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   str    
 7   EmployeeCount            4410 non-null   int64  
 8   EmployeeID               4410 non-null   int64  
 9   Gender                   4410 non-null   str    
 10  JobLevel                 4410 non-null   int64  
 11  JobRole                  4410 non-null   str    
 12  MaritalStatus            4410 non-null   str    
 13  MonthlyIncome            4410 non-null   int64  
 14  NumCompaniesWorked       4391 non-n

np.int64(0)

In [18]:
# Employee
df_employee.info()

df_employee.describe(include='all')

df_employee.isnull().sum()

df_employee.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EmployeeID               4410 non-null   int64  
 1   EnvironmentSatisfaction  4385 non-null   float64
 2   JobSatisfaction          4390 non-null   float64
 3   WorkLifeBalance          4372 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 137.9 KB


np.int64(0)

In [19]:
# Manager
df_manager.info()

df_manager.describe(include='all')

df_manager.isnull().sum()

df_manager.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   EmployeeID         4410 non-null   int64
 1   JobInvolvement     4410 non-null   int64
 2   PerformanceRating  4410 non-null   int64
dtypes: int64(3)
memory usage: 103.5 KB


np.int64(0)

In [20]:
# In time
df_inTime.info()

df_inTime.describe(include='all')

df_inTime.isnull().sum()

df_inTime.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Columns: 262 entries, Unnamed: 0 to 2015-12-31
dtypes: float64(12), int64(1), str(249)
memory usage: 8.8 MB


np.int64(0)

In [ ]:
# Out time
df_outTime.info()

df_outTime.describe(include='all')

df_outTime.isnull().sum()

df_outTime.duplicated().sum()

## Data Loading and Integration

In [24]:
df_final = df_general.merge(df_employee, on='EmployeeID').merge(df_manager, on='EmployeeID')

In [25]:
df_final.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,...,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,13.0,5,8,7,5,4.0,4.0,3.0,2,3
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,9.0,2,6,0,4,4.0,1.0,3.0,3,3


In [26]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   str    
 2   BusinessTravel           4410 non-null   str    
 3   Department               4410 non-null   str    
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   str    
 7   EmployeeCount            4410 non-null   int64  
 8   EmployeeID               4410 non-null   int64  
 9   Gender                   4410 non-null   str    
 10  JobLevel                 4410 non-null   int64  
 11  JobRole                  4410 non-null   str    
 12  MaritalStatus            4410 non-null   str    
 13  MonthlyIncome            4410 non-null   int64  
 14  NumCompaniesWorked       4391 non-n